# Elife Lung Random Forest Binary Classifier predictions

Andrew E. Davidson  
aedaivds@ucsc.edu 2/18/25  

Copyright (c) 2020-2023, Regents of the University of California All rights reserved.   https://polyformproject.org/licenses/noncommercial/1.0.0  

* ref:    
    - extraCellularRNA/londonCalling2024/jupyterNotebooks/nanoporeAdenocarcinomaBinaryClassification.ipynb

In [1]:
import ipynbname

# use display() to print an html version of a data frame
# useful if dataFrame output is not generated by last like of cell
from IPython.display import display

import joblib
# import math
import numpy as np
import os
import pandas as pd
# pd.set_option('display.max_rows', None)

from sklearn.ensemble        import RandomForestClassifier

import sys

notebookName = ipynbname.name()
notebookPath = ipynbname.path()
notebookDir = os.path.dirname(notebookPath)

import logging
# loglevel = "INFO"
loglevel = "WARN"
# logFMT = "%(asctime)s %(levelname)s [thr:%(threadName)s %(name)s %(funcName)s() line:%(lineno)s] [%(message)s]"
logFMT = "%(asctime)s %(levelname)s %(name)s %(funcName)s() line:%(lineno)s] [%(message)s]"
logging.basicConfig(format=logFMT, level=loglevel)    
logger = logging.getLogger(notebookName)

In [2]:
# setting the python path allows us to run python scripts from using
# the CLI. 
ORIG_PYTHONPATH = os.environ['PYTHONPATH']

# deconvolutionModules = notebookPath.parent.joinpath("../../deconvolutionAnalysis/python/")
deconvolutionModules = notebookPath.parent.joinpath("../..")
print("deconvolutionModules: {}\n".format(deconvolutionModules))

PYTHONPATH = ORIG_PYTHONPATH + f':{deconvolutionModules}'
print("PYTHONPATH: {}\n".format(PYTHONPATH))

# intraExtraRNA_POCModules=notebookPath.parent.joinpath("../../intraExtraRNA_POC/python/src")
intraExtraRNA_POCModules=notebookPath.parent.joinpath("/private/home/aedavids/extraCellularRNA/deconvolutionAnalysis/python/tempus/jupyterNotebooks/../../../../intraExtraRNA_POC/python/src")
print("intraExtraRNA_POCModules: {}\n".format(intraExtraRNA_POCModules))

PYTHONPATH = PYTHONPATH + f':{intraExtraRNA_POCModules}'
print("PYTHONPATH: {}\n".format(PYTHONPATH))

os.environ["PYTHONPATH"] = PYTHONPATH
PYTHONPATH = os.environ["PYTHONPATH"]
print("PYTHONPATH: {}\n".format(PYTHONPATH))

# to be able to import our local python files we need to set the sys.path
# https://stackoverflow.com/a/50155834
sys.path.append( str(deconvolutionModules) )
sys.path.append( str(intraExtraRNA_POCModules) )
print("\nsys.path:\n{}\n".format(sys.path))

deconvolutionModules: /private/home/aedavids/extraCellularRNA/deconvolutionAnalysis/python/tempus/jupyterNotebooks/../..

PYTHONPATH: :/private/home/aedavids/extraCellularRNA/src:/private/home/aedavids/extraCellularRNA/deconvolutionAnalysis/python/tempus/jupyterNotebooks/../..

intraExtraRNA_POCModules: /private/home/aedavids/extraCellularRNA/deconvolutionAnalysis/python/tempus/jupyterNotebooks/../../../../intraExtraRNA_POC/python/src

PYTHONPATH: :/private/home/aedavids/extraCellularRNA/src:/private/home/aedavids/extraCellularRNA/deconvolutionAnalysis/python/tempus/jupyterNotebooks/../..:/private/home/aedavids/extraCellularRNA/deconvolutionAnalysis/python/tempus/jupyterNotebooks/../../../../intraExtraRNA_POC/python/src

PYTHONPATH: :/private/home/aedavids/extraCellularRNA/src:/private/home/aedavids/extraCellularRNA/deconvolutionAnalysis/python/tempus/jupyterNotebooks/../..:/private/home/aedavids/extraCellularRNA/deconvolutionAnalysis/python/tempus/jupyterNotebooks/../../../../intraExt

In [3]:
# import local 
from analysis.utilities import loadDictionary
from analysis.utilities import loadList

from models.mlUtilities import loadEncoder

# Load Random Forest Lung Binary Classifier Trained on Elife Data

In [4]:
modelOut = "/private/groups/kimlab/aedavids/elife/elifeBinaryRandomForestResults.out/model"
modelName = "elife-Lung-Cancer-Health-control-random-forest-GTEx-Lung-biomarkers"
modelPath = f"{modelOut}/{modelName}.joblib"
rfModel = joblib.load(modelPath)
print(f'loaded model: {modelPath}')

loaded model: /private/groups/kimlab/aedavids/elife/elifeBinaryRandomForestResults.out/model/elife-Lung-Cancer-Health-control-random-forest-GTEx-Lung-biomarkers.joblib


/private/home/aedavids/miniconda3/envs/extraCellularRNA/lib/python3.11/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.4.0 when using version 1.5.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/private/home/aedavids/miniconda3/envs/extraCellularRNA/lib/python3.11/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.4.0 when using version 1.5.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [5]:
featureNamesPath = f"{modelOut}/{modelName}_features.txt"
featureNames = loadList(featureNamesPath)
featureNames

['FPR3',
 'CSF3',
 'SLAMF8',
 'ENTPD2',
 'MAGEE1',
 'PCAT19',
 'GRIP2',
 'PTGIR',
 'RND1',
 'CHRNB1']

In [6]:
encoderPath = f'{modelOut}/{modelName}.labelEncoder.txt'
labelEncoder = loadEncoder(encoderPath)
classes = labelEncoder.classes_
print(f'labelEncoder classes :\n{classes}')

labelEncoder classes :
['Healthy donor' 'Lung Cancer']


# Load the count data

In [7]:
dataRoot = "/private/groups/kimlab/aedavids/elife/createGTEx_TCGA_elifeDataSets.out/data"
mapDFPath = f'{dataRoot}/mapDF.csv'
mapDF = pd.read_csv(mapDFPath)
print(f'load {mapDFPath}')

load /private/groups/kimlab/aedavids/elife/createGTEx_TCGA_elifeDataSets.out/data/mapDF.csv


In [8]:
print(f'mapDF.shape : {mapDF.shape}')
mapDF.iloc[0:5, 0:6]

mapDF.shape : (70, 3)


,HUGO_v35,ENSG_v35,ENSG_v39
0,EBNA1BP2,ENSG00000117395.13,ENSG00000117395.13
1,SLAMF8,ENSG00000158714.11,ENSG00000158714.11
2,YOD1,ENSG00000180667.10,ENSG00000180667.11
3,SLC30A1,ENSG00000170385.10,ENSG00000170385.10
4,ROCK2,ENSG00000134318.14,ENSG00000134318.15


In [10]:
countFile = "countDF.csv"
countPath = f'{dataRoot}/{countFile}'
countDF = pd.read_csv(countPath)

In [11]:
print(f'countDF.shape : {countDF.shape}')
countDF.iloc[0:5, 0:6]

countDF.shape : (224, 76555)


,(A)n,(AAA)n,(AAAAAAC)n,(AAAAAAG)n,(AAAAAAT)n,(AAAAAC)n
0,201.672053,0.0,0.0,0.0,0.0,0.0
1,110.450773,0.0,0.0,0.0,0.0,0.0
2,3722.776395,0.0,0.0,0.0,0.0,0.0
3,1394.605651,0.0,0.0,0.0,0.0,0.0
4,2843.473730,0.0,0.0,0.0,0.0,0.0


In [ ]:
# countDF.loc[:, featureNames]

In [12]:
for hugoName in featureNames:
    if not hugoName in countDF.columns:
        print(f'!!!! missing {hugoName}')
    else:
        print(f'found {hugoName}')

!!!! missing FPR3
!!!! missing CSF3
!!!! missing SLAMF8
!!!! missing ENTPD2
!!!! missing MAGEE1
!!!! missing PCAT19
!!!! missing GRIP2
!!!! missing PTGIR
!!!! missing RND1
!!!! missing CHRNB1


In [20]:
xxxL = mapDF['HUGO_v35'].to_list()
for hugoName in featureNames:
    if not hugoName in xxxL:
        print(f'!!!! missing {hugoName}')
    else:
        print(f'found {hugoName}')

found FPR3
found CSF3
found SLAMF8
found ENTPD2
found MAGEE1
found PCAT19
found GRIP2
found PTGIR
found RND1
found CHRNB1
